In [1]:
import pandas as pd
import statsmodels.api as sm
from scipy import stats
import sys
sys.path.append(rf"/Users/baia/Desktop/PYTHON/mba_dsa_usp_esalq")

from TCC.utils.constantes import *

In [ ]:
# ==========================================
# 1. DEFINIÇÃO DOS BLOCOS DE VARIÁVEIS
# ==========================================

# Variável dependente (Alvo)
Y_col = 'btc_log_ret'

# Grupo de Controle (Variáveis seguras, sem multicolinearidade severa)
# Exemplo: DXY, VIX, Ouro, Juros de 10 anos. (Altere com os nomes reais das suas colunas)
X_controle = [
    'dxy_log_ret', 
    'vix_log_ret', 
    'gold_log_ret', 
    'us10y_diff'
]

# Grupo Colinear (As variáveis "gêmeas" que não podem ficar juntas na mesma regressão)
X_colineares = [
    'spx_log_ret', 
    'nasdaq_log_ret'
    # Se tivesse o Dow Jones, adicionaria aqui: 'dow_log_ret'
]

# ==========================================
# 2. FUNÇÃO DO TESTE DE CHOW
# ==========================================
def rodar_teste_chow(df_total, df_1, df_2, y_name, x_names):
    """
    Roda o Teste de Quebra Estrutural de Chow para um conjunto específico de variáveis.
    """
    # Prepara as variáveis X adicionando a constante (intercepto)
    X_tot = sm.add_constant(df_total[x_names])
    X_v   = sm.add_constant(df_1[x_names])
    X_i   = sm.add_constant(df_2[x_names])
    
    # Roda as regressões MQO (OLS)
    mod_tot = sm.OLS(df_total[y_name], X_tot).fit()
    mod_v   = sm.OLS(df_1[y_name], X_v).fit()
    mod_i   = sm.OLS(df_2[y_name], X_i).fit()
    
    # Extrai a Soma dos Quadrados dos Resíduos (SSR)
    ssr_total = mod_tot.ssr
    ssr_1     = mod_v.ssr
    ssr_2     = mod_i.ssr
    
    # Parâmetros da fórmula
    k = len(x_names) + 1  # Número de variáveis explicativas + 1 (constante)
    n1 = len(df_1)
    n2 = len(df_2)
    
    # Cálculo da Estatística F e P-valor
    numerador = (ssr_total - (ssr_1 + ssr_2)) / k
    denominador = (ssr_1 + ssr_2) / (n1 + n2 - 2 * k)
    
    f_stat = numerador / denominador
    p_value = stats.f.sf(f_stat, k, (n1 + n2 - 2 * k))
    
    return f_stat, p_value

# ==========================================
# 3. EXECUÇÃO DO RODÍZIO (LOOP)
# ==========================================
print("=== RESULTADOS DOS TESTES DE CHOW (RODÍZIO DE VARIÁVEIS COLINEARES) ===\n")

# Para cada variável colinear na nossa lista, criamos um modelo e testamos
for var_colinear in X_colineares:
    # O modelo atual será as variáveis de controle + 1 variável colinear da vez
    modelo_atual_X = X_controle + [var_colinear]
    
    print(f"Testando Modelo com: {var_colinear.upper()} + Variáveis de Controle")
    print(f"Regressores exatos: {modelo_atual_X}")
    
    # Roda a função do Teste de Chow (Assumindo que seus dataframes se chamam df, df_varejo e df_institucional)
    f_stat, p_val = rodar_teste_chow(df, df_varejo, df_institucional, Y_col, modelo_atual_X)
    
    # Imprime o resultado formatado
    print(f"Estatística F: {f_stat:.4f}")
    print(f"P-Valor:       {p_val:.10f}")
    
    # Avalia a quebra estrutural (considerando nível de significância de 5%)
    if p_val < 0.05:
        print("Conclusão:     QUEBRA ESTRUTURAL CONFIRMADA (Rejeita H0)")
    else:
        print("Conclusão:     NÃO HOUVE QUEBRA (Aceita H0)")
    print("-" * 70)